# Startup Distribution in Portugal

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
sys.path.append(os.path.abspath(".."))
from unidecode import unidecode
import importlib
import src.maps
from IPython.display import Image
importlib.reload(src.maps)
from src.maps import get_geojson, plot_choropleth, get_gpd_dataframe

In [ ]:
%run maps.ipynb #chart notebooks

In [ ]:
#Global visual style
sns.set_style("whitegrid")

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.titlesize": 18,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12
})

## 1. Introduction

Portugal's startup ecosystem has experienced significant growth in recent years, particularly around major urban centers such as Lisbon and Porto. This project explores the geographic distribution of startups across Portuguese municipalities and investigates whether socio-economic characteristics help explain these patterns.

## 2. Research Questions and Assumptions

The project is guided by the following research questions:

- How are startups distributed across Portugal?
- Is there significant geographic concentration?
- Which municipalities over or underperform?
- Which socio-economic factors appear associated with startups presence?

Initial assumptions:

- Startup activity is likely concentrated in Lisbon and Porto.
- Municipalities with higher purchasing power may attract more startups.
- Higher education levels may be associated with startup density. 

## 3. Data Sources

The project combines startup data from Dealroom with socio-economic indicators collected from PORDATA.

### Startup Data
- Municipality
- Funding
- Industry sector
- Founding year

### Socio-economic Data
- Population
- Population density
- Higher education rates
- Purchasing power
- Business creation metrics

## 4. Data Cleaning and Preparation

In [ ]:
#Import startups dataset
startups_dataframe = pd.read_csv("../data/processed_data/startups_clean.csv")
print(startups_dataframe.head(20))

In [ ]:
#Importing unemployment dataset
socio_unemployment10k_dataframe = pd.read_csv("../data/processed_data/02_df_unemployment_p10k.csv")
print(socio_unemployment10k_dataframe.head())

In [ ]:
#Importing new company dataset
socio_new_company10k_dataframe = pd.read_csv("../data/processed_data/03_df_company_p10k.csv")
print(socio_new_company10k_dataframe.head())

In [ ]:
#Importing education dataset
socio_university10k_dataframe = pd.read_csv("../data/processed_data/04_df_university_p10k.csv")
print(socio_university10k_dataframe.head())

### Checking the shape of the dataframes

In [ ]:
#Startups dataframe shape
startups_dataframe.shape

In [ ]:
#unemployment shape
socio_unemployment10k_dataframe.shape


In [ ]:
#new company shape
socio_new_company10k_dataframe.shape

In [ ]:
#university shape
socio_university10k_dataframe.shape

### Checking the info of the dataframes

In [ ]:
#Startup info
startups_dataframe.info()

In [ ]:
#unemployment info
socio_unemployment10k_dataframe.info()

In [ ]:
#new company info
socio_new_company10k_dataframe.info()

In [ ]:
#university info
socio_university10k_dataframe.info()

### District names normalization in Startup Dataset

In [ ]:
# normalize district names for startups
def normalize_district_names(column):
    return column.str.strip().str.title()  # Remove leading/trailing whitespace and convert to title case

#startup district names
startups_district_names_column = normalize_district_names(startups_dataframe['hq_state'])
#print(startups_district_names_column.head(20))


In [ ]:
# normalize district names for unemployment data
socio_unemployment_district_names_column = normalize_district_names(socio_unemployment10k_dataframe['territory'])
print(socio_unemployment_district_names_column.head(20))

In [ ]:
# normalize district names for new company data data
socio_new_company_district_names_column = normalize_district_names(socio_new_company10k_dataframe['territory'])
print(socio_new_company_district_names_column.head(20))

In [ ]:
# normalize district names for university data
socio_university_district_names_column = normalize_district_names(socio_university10k_dataframe['territory'])
print(socio_university_district_names_column.head(20))

In [ ]:
#Checking regions in the startups dataset that are not in the socioeconomic datasets
print(sorted(set(startups_district_names_column.dropna()) - (set(socio_unemployment_district_names_column.dropna()))))
print("\n")
print(sorted(set(startups_district_names_column.dropna()) - (set(socio_new_company_district_names_column.dropna()))))
print("\n")
print(sorted(set(startups_district_names_column.dropna()) - (set(socio_university_district_names_column.dropna()))))

In [ ]:
#checking regions that are in the socioeconomic datasets but not in the startups dataset
print(sorted(set(socio_unemployment_district_names_column) - (set(startups_district_names_column))))
print("\n")
print(sorted(set(socio_new_company_district_names_column) - (set(startups_district_names_column))))
print("\n")
print(sorted(set(socio_university_district_names_column) - (set(startups_district_names_column))))

### Checking unique and nunique municipalities

In [ ]:
#checking unique and nunique startups
print("Number of unique districts in startups dataset:", startups_district_names_column.nunique())
print(f"they are, {startups_district_names_column.unique()}")
print("\n")
print("So it's missing in all the socio-economic datasets:", sorted(set(startups_district_names_column.dropna()) - (set(socio_unemployment_district_names_column.dropna()))))


In [ ]:
#checking unique and nunique socioeconomic datasets
print("Number of unique districts in unemployment dataset:", socio_unemployment_district_names_column.nunique())
print(f"they are, {socio_unemployment_district_names_column.unique()}")
print("\n")
print("Number of unique districts in new company dataset:", socio_new_company_district_names_column.nunique())
print(f"they are, {socio_new_company_district_names_column.unique()}")
print("\n")
print("Number of unique districts in university dataset:", socio_university_district_names_column.nunique())
print(f"they are, {socio_university_district_names_column.unique()}")
print("\n")

print("So it's missing in startups dataset:", sorted(set(socio_unemployment_district_names_column.dropna()) - (set(startups_district_names_column.dropna()))))
print("A total of", len(sorted(set(socio_unemployment_district_names_column.dropna()) - (set(startups_district_names_column.dropna())))), "districts are missing in startups dataset compared to socio-economic datasets.")

### Geographic Granularity

> The startup dataset was available at district/regional level rather than municipality level. Therefore, socio-economic datasets were aggregated to the same geographic granularity to ensure consistency across analyses.

### Normalize municipality names for socioeconomic data at District Level

In [ ]:
# importing the dataset and extract just the municipality names from the startups dataset to compare with socio-economic datasets
nuts_dataframe = pd.read_csv("../data/processed_data/00_df_nuts.csv")
nuts_dataframe.head()

In [ ]:
#Checking the shape of the nuts dataframe
nuts_dataframe.shape

In [ ]:
#Checking the info of the nuts dataframe
nuts_dataframe.info()

In [ ]:
#Copy the nuts dataframe and extract just the region column
nuts_region_dataframe = nuts_dataframe.copy()
nuts_region_dataframe = nuts_region_dataframe[['region', 'territory']]
nuts_region_dataframe.head()

In [ ]:
#region column normalization
def normalize_region_names(columns):
    return columns.apply(lambda x: x.str.strip().str.title())  # Remove leading/trailing whitespace and convert to title case
nuts_region_dataframe[['region', 'territory']] = normalize_region_names(nuts_region_dataframe[['region', 'territory']])
nuts_region_dataframe.head()

In [ ]:
# We already have 2 region columns on the other socio-economic datasets (region_x and region_y) with a lot of missing values, so we will remove both and take only the region column from the nuts dataset to merge with the socio-economic datasets.
#socio_unemployment10k_dataframe["region_x"].isna().sum() #commented because they were already removed.
#socio_unemployment10k_dataframe["region_y"].isna().sum() #commented because they were already removed.
socio_unemployment10k_dataframe = (
    socio_unemployment10k_dataframe
    .drop(columns=["region_x", "region_y"], errors="ignore")
)


In [ ]:
socio_new_company10k_dataframe = (
    socio_new_company10k_dataframe
    .drop(columns=["region_x", "region_y"], errors="ignore")
)

In [ ]:
socio_university10k_dataframe = (
    socio_university10k_dataframe
    .drop(columns=["region"], errors="ignore")
)

In [ ]:
#Checking that both columns were removed
print("Unemployment dataset columns:", socio_unemployment10k_dataframe.columns)

In [ ]:
#Checking that both columns were removed
print("New company dataset columns:", socio_new_company10k_dataframe.columns)


In [ ]:
# checking that column "region" was removed
print("University dataset columns:", socio_university10k_dataframe.columns)

In [ ]:
#Normalize Territory column in the socio-economic datasets to match the territory column in the nuts dataset
socio_unemployment10k_dataframe['territory'] = socio_unemployment10k_dataframe['territory'].str.strip().str.title()

In [ ]:
socio_new_company10k_dataframe['territory'] = socio_new_company10k_dataframe['territory'].str.strip().str.title()

In [ ]:
socio_university10k_dataframe['territory'] = socio_university10k_dataframe['territory'].str.strip().str.title()

## 5. Feature Engineering

In [ ]:
#Merging NUTS data into socioeconomic dataframes to get the region information for each municipality

socio_unemployment10k_dataframe = (
    socio_unemployment10k_dataframe.merge(
        nuts_region_dataframe[["territory", "region"]],
        on="territory",
        how="left"
    )
)
socio_unemployment10k_dataframe.head()


In [ ]:
socio_new_company10k_dataframe = (
    socio_new_company10k_dataframe.merge(
        nuts_region_dataframe[["territory", "region"]],
        on="territory",
        how="left"
    )
)

socio_new_company10k_dataframe.head()

In [ ]:
socio_university10k_dataframe = (
    socio_university10k_dataframe.merge(
        nuts_region_dataframe[["territory", "region"]],
        on="territory",
        how="left"
    )
)

socio_university10k_dataframe.head()


In [ ]:
#reorder the columns so Territory and Region are next to each other
cols = socio_unemployment10k_dataframe.columns.tolist()
new_order = ['territory', 'region'] + [col for col in cols if col not in ['territory', 'region']]
socio_unemployment10k_dataframe = socio_unemployment10k_dataframe[new_order]
socio_unemployment10k_dataframe.head()

In [ ]:
#reorder the columns so Territory and Region are next to each other
cols2 = socio_new_company10k_dataframe.columns.tolist()
new_order2 = ['territory', 'region'] + [col for col in cols2 if col not in ['territory', 'region']]
socio_new_company10k_dataframe = socio_new_company10k_dataframe[new_order2]
socio_new_company10k_dataframe.head()

In [ ]:
#reorder the columns so Territory and Region are next to each other
cols3 = socio_university10k_dataframe.columns.tolist()
new_order3 = ['territory', 'region'] + [col for col in cols3 if col not in ['territory', 'region']]
socio_university10k_dataframe = socio_university10k_dataframe[new_order3]
socio_university10k_dataframe.head()

In [ ]:
#checking null values in the region column for unemployment dataset after the merge
print("Null values in region column of unemployment dataset:", socio_unemployment10k_dataframe['region'].isna().sum())

In [ ]:
#checking null values in the region column for new company dataset after the merge
print("Null values in region column of new company dataset:", socio_new_company10k_dataframe['region'].isna().sum())

In [ ]:
#checking null values in the region column for university dataset after the merge
print("Null values in region column of university dataset:", socio_university10k_dataframe['region'].isna().sum())

### Filtering socio-economic datasets by year 2023

> To ensure temporal consistency across datasets, the analysis focused on 2023, the latest year consistently available across the startup and socio-economic data sources.

In [ ]:
#Filtering unemployment dataset to year 2023
socio_unemployment10k_dataframe = (
    socio_unemployment10k_dataframe[
        socio_unemployment10k_dataframe["year"] == 2023
    ]
)

socio_unemployment10k_dataframe.head()

In [ ]:
#Filtering new company dataset to year 2023
socio_new_company10k_dataframe = (
    socio_new_company10k_dataframe[
        socio_new_company10k_dataframe["year"] == 2023
    ]
)

socio_new_company10k_dataframe.head()

In [ ]:
#Filtering education dataset to year 2023
socio_university10k_dataframe = (
    socio_university10k_dataframe[
        socio_university10k_dataframe["year"] == 2023
    ]
)

socio_university10k_dataframe.head()

### Normalizing again the filtered socio-economic datasets by year 2023

In [ ]:
#Normalize Territory column in the socio-economic datasets filtered to year 2023 to match the territory column in the nuts dataset
socio_unemployment10k_dataframe["territory"] = socio_unemployment10k_dataframe["territory"].str.strip().str.title()

In [ ]:
#Normalize Territory column in the socio-economic datasets filtered to year 2023 to match the territory column in the nuts dataset
socio_new_company10k_dataframe["territory"] = socio_new_company10k_dataframe["territory"].str.strip().str.title()

In [ ]:
#Normalize Territory column in the socio-economic datasets filtered to year 2023 to match the territory column in the nuts dataset
socio_university10k_dataframe["territory"] = socio_university10k_dataframe["territory"].str.strip().str.title()

In [ ]:
#removing region column region because it was already created previously.
socio_unemployment10k_dataframe = (
    socio_unemployment10k_dataframe
    .drop(columns=["region"], errors="ignore")
)

In [ ]:
#Merging NUTS data into socioeconomic dataframes filtered to year 2023 to get the region information for each municipality
socio_unemployment10k_dataframe = (
    socio_unemployment10k_dataframe.merge(
        nuts_region_dataframe[["territory", "region"]],
        on="territory",
        how="left"
    )
)

socio_unemployment10k_dataframe.head()

In [ ]:
#removing region column region because it was already created previously.
socio_new_company10k_dataframe = (
    socio_new_company10k_dataframe
    .drop(columns=["region"], errors="ignore")
)

In [ ]:
#Merging NUTS data into socioeconomic dataframes filtered to year 2023 to get the region information for each municipality
socio_new_company10k_dataframe = (
    socio_new_company10k_dataframe.merge(
        nuts_region_dataframe[["territory", "region"]],
        on="territory",
        how="left"
    )
)

socio_new_company10k_dataframe.head()

In [ ]:
#removing region column region because it was already created previously.
socio_university10k_dataframe = (
    socio_university10k_dataframe
    .drop(columns=["region"], errors="ignore")
)

In [ ]:
#Merging NUTS data into socioeconomic dataframes filtered to year 2023 to get the region information for each municipality
socio_university10k_dataframe = (
    socio_university10k_dataframe.merge(
        nuts_region_dataframe[["territory", "region"]],
        on="territory",
        how="left"
    )
)

socio_university10k_dataframe.head()

### Aggregating socio-economic datasets by region

In [ ]:
#Total Unemployment aggregation by region in 2023
regional_unemployment = (
    socio_unemployment10k_dataframe
    .groupby(["region"])
    .agg({
        "unemployment_total": "mean"
    })
    .reset_index()
)

sorted_regional_unemployment = regional_unemployment.sort_values(by="unemployment_total", ascending=False)
sorted_regional_unemployment.head()

In [ ]:
#New company aggregation by region in 2023
regional_new_company = (
    socio_new_company10k_dataframe
    .groupby(["region"])
    .agg({
        "company_births_total": "mean"
    })
    .reset_index()
)

sorted_regional_new_company = regional_new_company.sort_values(by="company_births_total", ascending=False)
sorted_regional_new_company.head()
    

In [ ]:
#Education level aggregation by region in 2023
regional_education = (
    socio_university10k_dataframe
    .groupby(["region"])
    .agg({
        "university_total": "mean"
    })
    .reset_index()
)

sorted_regional_education = regional_education.sort_values(by="university_total", ascending=False)
sorted_regional_education.head(10)


In [ ]:
# Load the population dataset (contains purchasing power per capita at municipality level)
population_df = pd.read_csv("../data/processed_data/01_df_population.csv")

# Standardise territory names so they match the other datasets and the NUTS map
population_df["territory"] = population_df["territory"].str.strip().str.title()

# Keep only 2023, the reference year used across the analysis
population_df = population_df[population_df["year"] == 2023].copy()

# Attach region to each municipality via the NUTS map 
population_df = population_df.merge(
    nuts_region_dataframe[["territory", "region"]], on="territory", how="left"
)
# Aggregate to region using mean
regional_purchasing_power = (
    population_df
    .groupby("region")
    .agg({"population_pp_pc": "mean"})
    .reset_index()
    .rename(columns={"population_pp_pc": "purchasing_power"})
)
regional_purchasing_power.head()

In [ ]:
regional_purchasing_power["region"] = regional_purchasing_power["region"].replace({
    "Região Autónoma Dos Açores": "Açores",
    "Região Autónoma Da Madeira": "Madeira",
    "Viana Do Castelo": "Viana do Castelo",
})

### Creating a Master DataFrame

In [ ]:
#Changing the region column name from the startups dataset to match the region column name in the socio-economic datasets
startups_dataframe = startups_dataframe.rename(columns={"hq_state": "region"})
startups_dataframe.head()

In [ ]:
#identifying and backfilling missing values on  startups funding column
startups_dataframe["total_funding_eur_m"].fillna(0, inplace=True)
startups_dataframe.head()

In [ ]:
#renaming the name column.
startups_dataframe = startups_dataframe.rename(columns={"name": "startup"})
startups_dataframe.head()

In [ ]:
#Creating an aggregated dataframe for startups.
df_aggregated_for_startups = (
    startups_dataframe
    .groupby("region")
    .agg({
        "startup": "count",
        "total_funding_eur_m": "sum"
    })
    .reset_index()
)

df_aggregated_for_startups

## Exploratory Data Analysis

### 6.1 Geographic Distribution

Lets create the Master Dataframe

In [ ]:
#First master dataframe, startups and unemployment
master_dataframe = (
    df_aggregated_for_startups.merge(
        regional_unemployment,
        on="region",
        how="left"
    )
)

master_dataframe.head()

In [ ]:
#concatenated master dataframe, startups and new company
master_dataframe = master_dataframe.merge(
    regional_new_company,
    on="region",
    how="left"
) 

master_dataframe.head()

In [ ]:
#concatenated master dataframe, startups and education
master_dataframe = master_dataframe.merge(
    regional_education,
    on="region",
    how="left"
)

master_dataframe.head()

In [ ]:
# Add regional purchasing power to the master dataframe
master_dataframe = master_dataframe.merge(
    regional_purchasing_power, 
    on="region", 
    how="left"
)

# Every region must receive a value. A mismatch would later be turned into 0 by fillna and appear as a false outlier — so we catch it here.
assert master_dataframe["purchasing_power"].notna().all(), \
    "Purchasing power merge produced missing values: region names do not align."

master_dataframe.head()

In [ ]:
#checking info on the master dataframe
master_dataframe.info()

In [ ]:
#Checking null values in the master dataframe
print("Null values in master dataframe:\n", master_dataframe.isna().sum())

In [ ]:
#removing duplicated columns year_x and year_y.
master_dataframe = master_dataframe.drop(columns=["year_x", "year_y"], errors="ignore")
master_dataframe.head()

In [ ]:
#checking shape
master_dataframe.shape

In [ ]:
#cleaning missing values.
master_dataframe = master_dataframe.fillna(0)
master_dataframe.head()

In [ ]:
#Lets start by understanding the data with all features, categorical and numerical.
master_dataframe.describe(include='all')

### First impressions

> 1. The startup ecosystem is very uneven. The average number of startups per region is 192, however, the region with the higher number os startups contains alone 1731 startups. This suggests a strong geographic concentration.
>
> 2. By looking at funding, we can see that this variable is even more concentrated. The average number of funding amounts is 394 million euros per region, but the top region concentrates 3305 million euros, 8 times the average amount. This suggests that not only exist more startups in certain regions, but also those companies get much more investment. The capital seems to be centralized
>
> 3. There is too much variability regarding startups and fundings. In both cases, the spread is much higher than the mean, suggesting that there is no balanced distribution, and some regions concentrate almost everything.
>
> 4. There are some regions with barely any startup activity. Regions where the minimum number of companies is 2 and the minimum funding investment amount is around 1 million euros. These first conclusions suggest territorial asymmetry.
>
> 5. Higher education indicates considerably variations across regions, with some regions displaying substantially stronger higher-education representation per 10.000 inhabitants. These findings are important to test one of our assumptions - "Startups tend to concentrate where higher education is stronger", and the data suggests that there are real differences that we should explore.
>
> 6. It seems there is less variability when it comes to unemployment, with the spread values being lower than the mean (122 std, 247 mean), so it is likely that the unemployment is not so correlated with more or less startups in a region, or its impact is weaker.

### 6.2 Socio-economic Patterns

#### Lets create a correlation matrix to understand the relationships between the numerical features in the master dataframe.

In [ ]:
correlation_matrix = master_dataframe.corr(numeric_only=True)
correlation_matrix

### Interpretation Rules

| correlation | meaning |
| ----------- | ------- |
| 0.7+        | strong  |
| 0.4 to 0.7  | moderate |
| 0.2 to 0.4  | weak    |
| ~0          | almost none |

> The strongest relationship is between startup count and total funding (r ≈ 0.94): regions with more startups also concentrate substantially more investment. This partly reflects how funding accumulates across many companies, so it should be read as co-concentration rather than an independent effect.
>
> University presence shows a moderate positive correlation with startups (r ≈ 0.40), consistent with our assumption that stronger higher-education ecosystems coincide with greater entrepreneurial activity.
>
> Purchasing power correlates with startups at r ≈ 0.70, but this is driven almost entirely by Lisbon and Porto; excluding them it weakens to r ≈ 0.36. Wealth therefore coincides with startup concentration in the two dominant metros rather than predicting it broadly.
>
> Unemployment shows essentially no relationship with startups (r ≈ 0.06) or with purchasing power (r ≈ −0.01). Notably, while unemployment correlates with general company creation (r ≈ 0.71), it does not correlate with startup formation — suggesting the drivers of ordinary business creation in higher-unemployment regions differ from those behind startup ecosystems.
>
> These associations are descriptive, not causal, and rest on roughly 20 regional observations, so individual coefficients should be read as indicative rather than precise.
And the Limitations sentence — this goes in cell 135 (your "## 10. Limitations" section), added as a new paragraph:
markdownThree regions (Açores, Madeira, Viana do Castelo) lacked matched values in some socio-economic sources and were treated as zero, which may slightly inflate correlations involving unemployment, company births, and university presence.
Quick placement recap so nothing lands in the wrong spot:

In [ ]:
plt.figure(figsize=(8,6))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Matrix")
plt.show()

_At the end of the day, we can say that startups are geographically concentrated and appear moderately associated with higher education ecosystems, while unemployment shows little relationship with startup concentration._

### 6.3 Visual Exploration of Key Relationships

The following visualizations help illustrate the relationships identified in the correlation analysis. The charts are divided into a Funding perpective and a Startup perpective.

### Funding vs University Presence

In [ ]:
plt.figure(figsize=(8,6))

sns.regplot(
    data=master_dataframe,
    x="university_total",
    y="total_funding_eur_m",
    scatter_kws={"s": 70},
    ci=None
)

plt.title("Funding vs University Presence")
plt.xlabel("University Presence")
plt.ylabel("Funding (Million EUR)")

plt.show()

### Funding vs Company Births

In [ ]:
plt.figure(figsize=(8,6))

sns.regplot(
    data=master_dataframe,
    x="company_births_total",
    y="total_funding_eur_m",
    scatter_kws={"s": 70},
    ci=None
)

plt.title("Funding vs Company Births")
plt.xlabel("Company Births")
plt.ylabel("Funding (Million EUR)")

plt.show()

### Funding vs Unemployment

In [ ]:
plt.figure(figsize=(8,6))

sns.regplot(
    data=master_dataframe,
    x="unemployment_total",
    y="total_funding_eur_m",
    scatter_kws={"s": 70},
    ci=None
)

plt.title("Funding vs Unemployment")
plt.xlabel("Unemployment")
plt.ylabel("Funding (Million EUR)")

plt.show()

### Startups vs Universities

In [ ]:
plt.figure(figsize=(8,6))

sns.regplot(
    data=master_dataframe,
    x="university_total",
    y="startup",
    ci=None
)

plt.title("Startups vs University Presence")
plt.xlabel("University Presence")
plt.ylabel("Number of Startups")

plt.show()

### Startups vs Company Births

In [ ]:
plt.figure(figsize=(8,6))

sns.regplot(
    data=master_dataframe,
    x="company_births_total",
    y="startup",
    ci=None
)

plt.title("Startups vs Company Births")
plt.xlabel("Company Births")
plt.ylabel("Number of Startups")

plt.show()

### Startup vs Unemployment

In [ ]:
plt.figure(figsize=(8,6))

sns.regplot(
    data=master_dataframe,
    x="unemployment_total",
    y="startup",
    ci=None
)

plt.title("Startups vs Unemployment")
plt.xlabel("Unemployment")
plt.ylabel("Number of Startups")

plt.show()

In [ ]:
# Relationship between regional purchasing power and startup presence.
# A positive slope supports the assumption that wealthier regions host more startups.
plt.figure(figsize=(8, 6))

sns.regplot(
    data=master_dataframe,
    x="purchasing_power",
    y="startup",
    scatter_kws={"s": 70},
    ci=None
)

plt.title("Startups vs Purchasing Power")
plt.xlabel("Purchasing Power per Capita")
plt.ylabel("Number of Startups")

plt.show()

### Interpretation of Relationships

> The regression plots visually reinforce several of the relationships identified previously in the correlation analysis.
> 
> A positive relationship can be observed between startup concentration and university presence, suggesting that regions with stronger higher education ecosystems also tend to present higher entrepreneurial activity. A similar, although weaker, pattern appears between startup concentration and general business creation.
> 
> On the other hand, unemployment does not appear to display a clear visual relationship with startup concentration, reinforcing the idea that startup ecosystems may depend more heavily on innovation capacity, investment attraction, and economic dynamism than on labor market conditions alone.
> 
> The plots also highlight the presence of significant territorial asymmetries, with Lisbon and Porto behaving as clear outliers when compared to the remaining Portuguese regions.


## 7. Geographic Analysis

### 7.1 Startup Geographic Distribution

### Startup Concentration Map

In [ ]:
#Keni data is set to work with municipalities yet. We already changed to regions in the rest of the project so for now, we will keep the municipality level and we will change it later. Lets create municipality_startups variable and use the "hq_city" column.
municipality_startups = (
    startups_dataframe
    .groupby("hq_city")
    .agg({
        "startup": "count"
    })
    .reset_index()
)

municipality_startups.columns = ["municipality", "startup_count"]

municipality_startups.head()

### Municipality Startup Normalization

In [ ]:
municipality_startups["municipality"] = (
    municipality_startups["municipality"]
    .str.upper()
    .apply(unidecode)
    .str.replace(" ", "-", regex=False)
)

### Marker Plots & Choropleth Map for Startups by Municipality

In [ ]:
plot_point_map(
    municipality_startups,
    label="municipality",
    metric="startup_count",
    title="Startup Distribution Across Portuguese Municipalities"
)

In [ ]:
plot_choropleth(
    municipality_startups,
    label="municipality",
    metric="startup_count",
    op="sum",
    title="Startup Concentration Across Portuguese Municipalities"
)

In [ ]:
# Image("../figures/choropleth_lisbon_startups.png")

### 7.2 Funding Geographic Distribution

### Funding Concentration Map

In [ ]:
municipality_funding = (
    startups_dataframe
    .groupby("hq_city")
    .agg({
        "total_funding_eur_m": "sum"
    })
    .reset_index()
)

municipality_funding.columns = [
    "municipality",
    "funding"
]

### Municipality Funding normalization

In [ ]:
municipality_funding["municipality"] = (
    municipality_funding["municipality"]
    .str.upper()
    .apply(unidecode)
    .str.replace(" ", "-", regex=False)
)

### Marker Plots & Choropleth Map for Fundings by Municipality

In [ ]:
plot_point_map(
    municipality_funding,
    label="municipality",
    metric="funding",
    title="Funding Distribution Across Portuguese Municipalities"
)

In [ ]:
plot_choropleth(
    municipality_funding,
    label="municipality",
    metric="funding",
    op="sum",
    title="Funding Distribution Across Portuguese Municipalities"
)

In [ ]:
# Image("../figures/choropleth_lisbon_fundings.png")

### Choropleth Map findings

> The choropleth maps reinforce the strong territorial concentration identified previously in the exploratory analysis.
> 
> Lisbon clearly dominates both startup concentration and funding attraction, while most municipalities display substantially lower levels of startup activity and investment.
> 
> These findings suggest that the Portuguese startup ecosystem remains highly centralized geographically.

## 8. Key Findings

1. The Portuguese startup ecosystem is highly concentrated geographically, with Lisbon and Porto clearly dominating both startup presence and funding attraction.

2. Funding concentration appears even stronger than startup concentration, suggesting that investment capital is heavily centralized in a small number of regions.

3. University presence shows a moderate positive relationship with startup concentration, supporting the idea that higher education ecosystems may contribute to entrepreneurial activity.

4. General business dynamism, measured through company births, also appears positively associated with startup concentration, although the relationship is weaker than expected. It gives an idea that everyone wants to be where everyone is.

5. Unemployment does not appear to have a strong relationship with startup concentration, suggesting that startup ecosystems may depend more on innovation infrastructure and investment environments than on labor market conditions.


## 9. Recommendations

Based on the findings of this analysis, some recommendations might be important for strengthening the Portuguese startup ecosystem across the country:

* Promote regional startup hubs outside Lisbon and Porto in order to reduce territorial concentration and stimulate local innovation ecosystems.

* Strengthen partnerships between universities and startups, particularly in regions with lower entrepreneurial activity.

* Improve access to funding opportunities in less represented regions to encourage startup creation and growth beyond the main urban centers.

* Support local business ecosystems and incubators capable of fostering entrepreneurial networks at the regional level.

* Continue investing in education, innovation, and technological infrastructure, as these factors appear positively associated with startup concentration.


## 10. Limitations

This project might present some limitations that should be considered when interpreting the findings.

First, the analysis relies on available public datasets, which may contain inconsistencies, missing values, or differences in territorial aggregation.

Second, the study focuses mainly on exploratory data analysis and correlation patterns rather than causal inference. Therefore, relationships identified between variables should not be interpreted as direct causal effects. Additionally, the analysis was conducted at a regional and municipal level, which may hide important local dynamics within regions.

Third, three regions (Açores, Madeira, Viana do Castelo) lacked matched values in some socio-economic sources and were treated as zero, which may slightly inflate correlations involving unemployment, company births, and university presence.

Finally, the study focuses on the Portuguese startup ecosystem in 2023 because it was the common year between all the datasets, meaning that results may evolve over time as economic and entrepreneurial conditions change.


## 11. Conclusion

This project explored the geographic distribution of startups across Portugal and analyzed several socio-economic factors potentially associated with startup concentration.

The analysis confirmed the initial assumption that startup activity is strongly concentrated in Lisbon and Porto. These regions not only host a substantially larger number of startups, but also attract a disproportionately high share of investment funding.

The findings suggest that university presence and broader business dynamism may contribute positively to startup ecosystems, while unemployment does not appear to play a significant role in explaining startup concentration.

Overall, the Portuguese startup ecosystem appears highly centralized geographically, reinforcing the importance of regional innovation policies capable of fostering entrepreneurial development beyond the main metropolitan areas.
